In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras import Model, layers
import gc
import matplotlib.pyplot as plt
import os
import keras


class CONFIG:
    seed = 42
    target_col = "responder_6"
    lag_cols_original = ["date_id", "symbol_id"] + [f"responder_{idx}" for idx in range(9)]
    lag_cols_rename = { f"responder_{idx}" : f"responder_{idx}_lag_1" for idx in range(9)}
    lag_cols = [f"responder_{idx}_lag_1" for idx in range(9)]
    windows = [484, 968]  # Half day, full day, 3 days
    created_features_names = (
        [f"{target_col}_{suffix}" for target_col in lag_cols 
         for suffix in [f"rolling_std_{window}" for window in [484, 968]] 
                       # +[f"momentum_{window}" for window in [484, 968]] 
                       #+ [f"ewm_mean_{window}" for window in [484, 968]]
        ]
    )
    time_features_names = ["sin_date_id", "cos_date_id", "sin_time_id", "cos_time_id"]
    valid_date = 1660
    original_features_names = [f'feature_{idx:02d}' for idx in range(79)] + [f'responder_{idx}_lag_1' for idx in range(9)]
    symbol_feature_name = ['symbol_id']
    target_name = 'responder_6'
    weight_name = 'weight'
    model_paths = ["/kaggle/input/js-xs-nn-trained-model"]

base_dir = "/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/"
partitions = [f"{base_dir}/partition_id={i}/part-0.parquet" for i in range(8, 10)]
train = pl.concat([pl.read_parquet(part) for part in partitions])

# Create lags
lags = train.select(pl.col(CONFIG.lag_cols_original))
lags = lags.rename(CONFIG.lag_cols_rename)
lags = lags.with_columns(
    date_id = pl.col('date_id') + 1,  # lagged by 1 day
)

# Create features
for col in CONFIG.lag_cols:
    for window in CONFIG.windows:
        lags = lags.with_columns([
            pl.col(col).rolling_std(window_size=window).over(['symbol_id']).clip(0, 5)
                .alias(f'{col}_rolling_std_{window}'),
            
            # Momentum captures trend
        #     ((pl.col(col) - pl.col(col).rolling_mean(window_size=window)) / 
        #      pl.col(col).rolling_std(window_size=window))
        #     .over(['symbol_id']).clip(-5, 5).alias(f'{col}_momentum_{window}')
        ])
print('Complete creating features')
    
lags = lags.group_by(["date_id", "symbol_id"], maintain_order=True).last()
train = train.join(lags, on=["date_id", "symbol_id"], how="left")
del lags
gc.collect()

# Add time features
train = train.with_columns([
    (2 * np.pi * pl.col('date_id') / 252).sin().alias('sin_date_id'),
    (2 * np.pi * pl.col('date_id') / 252).cos().alias('cos_date_id'),
    (2 * np.pi * pl.col('time_id') / 967).sin().alias('sin_time_id'),
    (2 * np.pi * pl.col('time_id') / 967).cos().alias('cos_time_id')
])
train = train.fill_null(strategy="forward")
train = train.drop_nulls()

val = train.filter(pl.col('date_id') >= CONFIG.valid_date)
train = train.filter(pl.col('date_id') < CONFIG.valid_date)

X_train_ori = train.select(CONFIG.original_features_names).to_pandas()
X_train_sym = train.select(CONFIG.symbol_feature_name).to_pandas()
X_train_time = train.select(CONFIG.time_features_names).to_pandas()
X_train_created = train.select(CONFIG.created_features_names).to_pandas()
y_train = train.select(CONFIG.target_name).to_pandas()
w_train = train.select(CONFIG.weight_name).to_pandas()
del train
gc.collect()

X_val_ori = val.select(CONFIG.original_features_names).to_pandas()
X_val_sym = val.select(CONFIG.symbol_feature_name).to_pandas()
X_val_time = val.select(CONFIG.time_features_names).to_pandas()
X_val_created = val.select(CONFIG.created_features_names).to_pandas()
y_val = val.select(CONFIG.target_name).to_pandas()
w_val = val.select(CONFIG.weight_name).to_pandas()
del val
gc.collect()

keras.config.enable_unsafe_deserialization()

def load_keras_models(model_dir='/kaggle/input/jane-street-convert-model/keras_models', num_folds=5):
    models = []
    for i in range(num_folds):
        try:
            model_path = os.path.join(model_dir, f'model_fold_{i}')
            model = tf.keras.models.load_model(model_path + '.keras')
            models.append(model)
        except Exception as e:
            print(f"Error loading model {i}: {str(e)}")
            raise e
    return models

models = load_keras_models()
print('Complete loading models')

class SAMOptimizer(tf.keras.optimizers.Optimizer):
    def __init__(self, optimizer, rho=0.05, name="SAM", **kwargs):
        if hasattr(optimizer.learning_rate, 'numpy'):
            init_lr = float(tf.keras.backend.get_value(optimizer.learning_rate))
        elif tf.is_tensor(optimizer.learning_rate):
            init_lr = float(tf.keras.backend.get_value(optimizer.learning_rate))
        else:
            init_lr = float(optimizer.learning_rate)
            
        super().__init__(learning_rate=init_lr, name=name, **kwargs)
        self.optimizer = optimizer
        self.rho = rho
        
    def _create_slots(self, var_list):
        self.optimizer._create_slots(var_list)
        
    def get_config(self):
        config = super().get_config()
        config.update({
            "rho": self.rho,
            "optimizer": tf.keras.optimizers.serialize(self.optimizer)
        })
        return config
    
    @property
    def learning_rate(self):
        if hasattr(self.optimizer.learning_rate, 'numpy'):
            return float(tf.keras.backend.get_value(self.optimizer.learning_rate))
        return self.optimizer.learning_rate
        
    @learning_rate.setter
    def learning_rate(self, value):
        if hasattr(self.optimizer.learning_rate, 'assign'):
            self.optimizer.learning_rate.assign(value)
        else:
            self.optimizer.learning_rate = value
    
    def _resource_apply_dense(self, grad, var, apply_state=None):
        try:
            # Clip gradients to prevent explosion
            grad = tf.clip_by_value(grad, -1.0, 1.0)
            
            # Check for NaN/Inf in gradients
            if tf.reduce_any(tf.math.is_nan(grad)) or tf.reduce_any(tf.math.is_inf(grad)):
                print(f"Warning: NaN/Inf detected in gradients for variable {var.name}")
                return self.optimizer._resource_apply_dense(grad, var, apply_state)
            
            grad_norm = tf.linalg.global_norm([grad])
            scale = self.rho / (grad_norm + 1e-12)
            
            orig_var = tf.identity(var)
            
            eps = grad * scale
            eps = tf.clip_by_value(eps, -0.1, 0.1)
            var.assign_add(eps)
            
            result = self.optimizer._resource_apply_dense(grad, var, apply_state)
            var.assign(orig_var)
            
            return result
        except Exception as e:
            print(f"Error in SAM optimizer: {str(e)}")
            return self.optimizer._resource_apply_dense(grad, var, apply_state)
    
    def _resource_apply_sparse(self, grad, var, indices, apply_state=None):
        return self.optimizer._resource_apply_sparse(grad, var, indices, apply_state)
    
    def get_gradients(self, loss, params):
        return self.optimizer.get_gradients(loss, params)
    
    def apply_gradients(self, grads_and_vars, **kwargs):
        return self.optimizer.apply_gradients(grads_and_vars, **kwargs)

class NanMonitorCallback(tf.keras.callbacks.Callback):
    def on_batch_end(self, batch, logs=None):
        logs = logs or {}
        for k, v in logs.items():
            if np.isnan(v) or np.isinf(v):
                print(f'\nNaN/Inf detected in {k} at batch {batch}')
                self.model.stop_training = True
                break


In [ ]:
# Add these checks before training
def check_data(X_train_ori, X_train_sym, X_train_time, X_train_created, y_train, w_train):
    print("Checking for NaN/Infinite values:")
    print(f"X_train_ori: {np.any(np.isnan(X_train_ori))} (NaN) {np.any(np.isinf(X_train_ori))} (Inf)")
    print(f"X_train_sym: {np.any(np.isnan(X_train_sym))} (NaN) {np.any(np.isinf(X_train_sym))} (Inf)")
    print(f"X_train_time: {np.any(np.isnan(X_train_time))} (NaN) {np.any(np.isinf(X_train_time))} (Inf)")
    print(f"X_train_created: {np.any(np.isnan(X_train_created))} (NaN) {np.any(np.isinf(X_train_created))} (Inf)")
    print(f"y_train: {np.any(np.isnan(y_train))} (NaN) {np.any(np.isinf(y_train))} (Inf)")
    print(f"w_train: {np.any(np.isnan(w_train))} (NaN) {np.any(np.isinf(w_train))} (Inf)")

check_data(X_train_ori, X_train_sym, X_train_time, X_train_created, y_train, w_train)

Try only fine tune the orignial model. Trainable false? loss?

In [ ]:
class EfficientNN:
    def __init__(
        self,
        pretrained_model,
        hidden_dim=256,
        dropout_rate=0.2,
        lr=1e-3,
        num_symbols=100,
        embedding_dim=32
    ):
        self.lr = lr
        self.pretrained_model = tf.keras.models.clone_model(pretrained_model)
        self.pretrained_model.set_weights(pretrained_model.get_weights())
        for layer in self.pretrained_model.layers:
            layer.trainable = False
            
        model = self.build_model(
            hidden_dim,
            dropout_rate,
            num_symbols,
            embedding_dim
        )
        
        optimizer = SAMOptimizer(
            optimizer=tf.keras.optimizers.AdamW(
                learning_rate=1e-4,
                clipnorm=1.0
            ),
            rho=0.02
        )
        
        model.compile(
            optimizer=optimizer,
            loss=self.weighted_huber_loss,
            metrics=[self.r2_score]
        )
        
        self.model = model
    
    def build_model(self, hidden_dim, dropout_rate, num_symbols, embedding_dim):

        original_input = layers.Input(shape=(len(CONFIG.original_features_names),), name='original_features')
        symbol_input = layers.Input(shape=(1,), name='symbol_id')        
        time_features_input = layers.Input(shape=(len(CONFIG.time_features_names),), name='time_features')
        created_features_input = layers.Input(shape=(len(CONFIG.created_features_names),), name='created_features')

        # Base prediction
        base_pred = self.pretrained_model(original_input)
        if isinstance(base_pred, list):
            base_pred = base_pred[0]
        
        # Symbol embedding
        symbol_embedding = layers.Embedding(
            input_dim=num_symbols,
            output_dim=embedding_dim,
            embeddings_regularizer=tf.keras.regularizers.l2(1e-4)
        )(symbol_input)
        symbol_embedding = layers.Flatten()(symbol_embedding)
        symbol_embedding = layers.Dense(hidden_dim)(symbol_embedding)
        
        # Time features
        time_features = layers.BatchNormalization()(time_features_input)
        time_features = layers.Dense(hidden_dim // 2, activation='selu')(time_features)
        
        # Created features
        created_features = layers.BatchNormalization()(created_features_input)
        created_features = layers.Dense(hidden_dim // 2, activation='selu')(created_features)
        
        # Feature interaction
        feature_interaction = layers.multiply([time_features, created_features])
        feature_sum = layers.add([time_features, created_features])
        
        # Combine features
        combined_features = layers.concatenate([
            feature_interaction,
            feature_sum,
            symbol_embedding,
            layers.Dense(hidden_dim)(base_pred)
        ])
        
        x = layers.Dense(hidden_dim, activation='selu')(combined_features)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.BatchNormalization()(x)
        
        residual = x
        x = layers.Dense(hidden_dim, activation='selu')(x)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.Dense(hidden_dim, activation='selu')(x)
        x = layers.Add()([x, residual])
        
        x = layers.Dense(hidden_dim // 2, activation='selu')(x)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.Dense(1, activation='tanh')(x)
        output = layers.Lambda(lambda x: 5 * x)(x)
        
        return Model(
            inputs={
                'original_features': original_input,
                'symbol_id': symbol_input,
                'time_features': time_features_input,
                'created_features': created_features_input
            },
            outputs=output
        )

    def weighted_huber_loss(self, y_true, y_pred, sample_weight=None, delta=1.0):
        eps = 1e-7
        
        error = tf.clip_by_value(y_true - y_pred, -10.0, 10.0)
        is_small_error = tf.abs(error) <= delta
        squared_loss = 0.5 * tf.square(error)
        linear_loss = delta * tf.abs(error) - 0.5 * tf.square(delta)
        loss = tf.where(is_small_error, squared_loss, linear_loss)
        
        if sample_weight is not None:
            sample_weight = tf.clip_by_value(sample_weight, eps, 1.0)
            loss *= sample_weight
            
        return tf.reduce_mean(loss) + eps

    @staticmethod
    def r2_score(y_true, y_pred, sample_weight=None):
        if sample_weight is None:
            sample_weight = tf.ones_like(y_true)
            
        weighted_mse = tf.reduce_sum(sample_weight * tf.square(y_true - y_pred))
        weighted_var = tf.reduce_sum(sample_weight * tf.square(y_true))
        
        r2 = 1 - (weighted_mse / (weighted_var + tf.keras.backend.epsilon()))
        return r2

    def fit(self, X_train_ori, X_train_sym, X_train_time, X_train_created, y_train, w_train, 
            X_val_ori=None, X_val_sym=None, X_val_time=None, X_val_created=None, y_val=None, w_val=None,
            batch_size=8192, epochs=100, patience=15):
        """
        Fit the model with separated time and created features
        """
        train_data = {
            'original_features': X_train_ori,
            'symbol_id': X_train_sym,
            'time_features': X_train_time,
            'created_features': X_train_created
        }
        
        validation_data = None
        if all(v is not None for v in [X_val_ori, X_val_sym, X_val_time, X_val_created, y_val]):
            validation_data = ({
                'original_features': X_val_ori,
                'symbol_id': X_val_sym,
                'time_features': X_val_time,
                'created_features': X_val_created
            }, y_val)
            if w_val is not None:
                validation_data = (*validation_data, w_val)
        
        callbacks_list = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=self.lr * 0.01
            ),
            tf.keras.callbacks.ModelCheckpoint(
                filepath='best_model.keras',
                monitor='val_loss',
                save_best_only=True
            )
        ]
        callbacks_list.append(NanMonitorCallback())
        
        history = self.model.fit(
            train_data,
            y_train,
            sample_weight=w_train,
            validation_data=validation_data,
            batch_size=batch_size,
            epochs=epochs,
            callbacks=callbacks_list,
            verbose=1
        )
        
        return history
        
    def predict(self, X_ori, X_sym, X_time, X_created):
        """Make predictions using the model"""
        predictions = self.model.predict({
            'original_features': X_ori,
            'symbol_id': X_sym,
            'time_features': X_time,
            'created_features': X_created
        })
        return predictions

def train_and_visualize_models(X_train_ori, X_train_sym, X_train_time, X_train_created,
                             X_val_ori, X_val_sym, X_val_time, X_val_created,
                             y_train, y_val, w_train, w_val, models):
    
    fine_tuned_models = []
    histories = []
    
    for i, pretrained_model in enumerate(models):
        print(f"\nTraining model {i+1}/{len(models)}")
        
        model = EfficientNN(
            pretrained_model=pretrained_model,
            hidden_dim=256,
            dropout_rate=0.2,
            lr=1e-4 
        )
        
        history = model.fit(
            X_train_ori=X_train_ori,
            X_train_sym=X_train_sym,
            X_train_time=X_train_time,
            X_train_created=X_train_created,
            y_train=y_train,
            w_train=w_train,
            X_val_ori=X_val_ori,
            X_val_sym=X_val_sym,
            X_val_time=X_val_time,
            X_val_created=X_val_created,
            y_val=y_val,
            w_val=w_val,
            batch_size=8192,
            epochs=1
        )

        val_predictions = model.predict(
            X_val_ori,
            X_val_sym,
            X_val_time,
            X_val_created
        )
        
        results = pd.DataFrame({
            'symbol_id': X_val_sym.values.flatten(),
            'val_predictions': val_predictions.flatten(),
            'y_val': y_val.values.flatten()
        })
        
        for symbol_id in range(4):
            symbol_data = results[results['symbol_id'] == symbol_id].tail(968*2)
            plt.figure(figsize=(15, 10))
            plt.plot(symbol_data['val_predictions'], 
                    label=f'Pred (sym={symbol_id})', 
                    alpha=0.7)
            plt.plot(symbol_data['y_val'], 
                    label=f'Actual (sym={symbol_id})', 
                    alpha=0.7)
            
            plt.title(f'Symbol ID {symbol_id}')
            plt.xlabel('Time')
            plt.ylabel('Values')
            plt.legend()
            plt.grid(True)
            plt.show()
        
        r2 = EfficientNN.r2_score(y_val.values.flatten(), val_predictions.flatten(), w_val.values.flatten())
        print(f"R2 Score: {r2:.4f}")
    
        model.model.save(f'fine_tuned_model_{i}.keras')
        
        fine_tuned_models.append(model)
        histories.append(history)
        
    return fine_tuned_models, histories


fine_tuned_models, histories = train_and_visualize_models(
    X_train_ori, X_train_sym, X_train_time, X_train_created,
    X_val_ori, X_val_sym, X_val_time, X_val_created,
    y_train, y_val, w_train, w_val, models
)